In [1]:
# 01_data_pipeline.ipynb
# 목적: 주가 데이터 + 거시경제 데이터 수집 및 저장

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader as pdr
from dotenv import load_dotenv
import os
from dotenv import load_dotenv

env_path = os.path.expanduser("~/Desktop/SentriVaR-500/.env")
load_dotenv(env_path)
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

START_DATE = "2018-01-01"
TICKERS = ["AAPL", "MSFT", "GOOGL", "JPM", "SOXX"]

print(f"API 키 확인: {NEWSAPI_KEY[:8]}...")
print(f"종목: {TICKERS}")

API 키 확인: 243b95d4...
종목: ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'SOXX']


In [2]:
# 주가 데이터 수집
df = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)["Close"]
df = df.dropna()

print(f"데이터 크기: {df.shape}")
print(f"기간: {df.index[0].date()} ~ {df.index[-1].date()}")
df.tail()

$MSFT: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)
$AAPL: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)
$SOXX: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)
$JPM: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)
$GOOGL: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)

5 Failed downloads:
['MSFT', 'AAPL', 'SOXX', 'JPM', 'GOOGL']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-08)


데이터 크기: (0, 5)


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# 거시경제 데이터 수집
vix = pdr.get_data_fred("VIXCLS", start=START_DATE)
spread = pdr.get_data_fred("T10Y2Y", start=START_DATE)
cpi = pdr.get_data_fred("CPIAUCSL", start=START_DATE)

macro = pd.concat([vix, spread, cpi], axis=1)
macro.columns = ["VIX", "Spread", "CPI"]
macro = macro.dropna()

print(f"거시경제 데이터 크기: {macro.shape}")
macro.tail()

In [ ]:
# 거시경제 데이터 따로 저장 (주기가 달라서 합치지 않음)
# VIX, Spread = 일별 / CPI = 월별

macro_daily = pd.concat([vix, spread], axis=1, sort=False)
macro_daily.columns = ["VIX", "Spread"]
macro_daily = macro_daily.dropna()

macro_monthly = cpi.copy()
macro_monthly.columns = ["CPI"]

print(f"일별 거시경제 데이터: {macro_daily.shape}")
print(f"월별 CPI 데이터: {macro_monthly.shape}")
macro_daily.tail()

In [ ]:
# data 폴더 없으면 자동 생성 후 저장
import os

data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
os.makedirs(data_path, exist_ok=True)  # 폴더 없으면 자동 생성

df.to_csv(f"{data_path}/prices.csv")
macro_daily.to_csv(f"{data_path}/macro_daily.csv")
macro_monthly.to_csv(f"{data_path}/macro_monthly.csv")

print("저장 완료:")
print(f"  prices.csv        — {df.shape}")
print(f"  macro_daily.csv   — {macro_daily.shape}")
print(f"  macro_monthly.csv — {macro_monthly.shape}")